In [167]:
from pathlib import Path
from typing import Optional
import csv
import pandas as pd
import numpy as np
TARGET_YEAR = 2020
SMALL_HOLDING_USD = 500_000.0
# Appendix C offshore financial centers (investor side to avoid double counting).
OFC_ISO3 = {"BMU", "CYM", "GGY", "IRL", "IMN", "JEY", "LUX", "AN", "ANT"}
COUNTRY_ORDER = [
    "CAN", "USA", "AUT", "BEL", "DNK", "FIN", "FRA", "DEU", "ISR", "ITA", "NLD", "NOR", "PRT", "ESP", "SWE", "CHE", "GBR",
    "AUS", "HKG", "JPN", "NZL", "SGP", "BRA", "CHN", "COL", "CZE", "GRC", "HUN", "IND", "MYS", "MEX", "PHL", "POL", "RUS", "ZAF", "KOR", "THA",
]
EQUITY_SOURCE = {
    "CAN": "OECD", "USA": "OECD", "AUT": "OECD", "BEL": "OECD", "DNK": "OECD", "FIN": "OECD", "FRA": "OECD", "DEU": "OECD", "ISR": "OECD", "ITA": "OECD",
    "NLD": "OECD", "NOR": "OECD", "PRT": "OECD", "ESP": "OECD", "SWE": "OECD", "CHE": "OECD", "GBR": "OECD", "AUS": "WB", "HKG": "WB", "JPN": "OECD",
    "NZL": "WB", "SGP": "WB", "BRA": "OECD", "CHN": "WB", "COL": "OECD", "CZE": "OECD", "GRC": "OECD", "HUN": "OECD", "IND": "WB", "MYS": "WB",
    "MEX": "OECD", "PHL": "WB", "POL": "OECD", "RUS": "WB", "ZAF": "WB", "KOR": "OECD", "THA": "WB",
}
DEBT_SOURCE_BASE = {
    "CAN": "OECD", "USA": "OECD", "AUT": "OECD", "BEL": "OECD", "DNK": "OECD", "FIN": "OECD", "FRA": "OECD", "DEU": "OECD", "ITA": "OECD", "NLD": "OECD",
    "NOR": "OECD", "PRT": "OECD", "ESP": "OECD", "SWE": "OECD", "CHE": "OECD", "GBR": "OECD", "AUS": "BIS", "HKG": "BIS", "JPN": "OECD", "NZL": "BIS",
    "SGP": "BIS", "CHN": "BIS", "COL": "OECD", "CZE": "OECD", "GRC": "OECD", "HUN": "OECD", "IND": "BIS", "MYS": "BIS", "MEX": "OECD", "PHL": "BIS",
    "POL": "OECD", "RUS": "BIS", "ZAF": "BIS", "KOR": "OECD", "THA": "BIS",
}
DEBT_SOURCE_SWITCHES = {
    "BRA": {"from_year": 2009, "before": "BIS", "after": "OECD"},
    "ISR": {"from_year": 2010, "before": "BIS", "after": "OECD"},
}
SAMPLE_START_YEAR = {
    "CAN": 2003, "USA": 2003, "AUT": 2003, "BEL": 2003, "DNK": 2003, "FIN": 2003, "FRA": 2003, "DEU": 2003, "ISR": 2003, "ITA": 2003,
    "NLD": 2003, "NOR": 2003, "PRT": 2003, "ESP": 2003, "SWE": 2003, "CHE": 2003, "GBR": 2003, "AUS": 2003, "HKG": 2003, "JPN": 2003,
    "NZL": 2003, "SGP": 2003, "BRA": 2003, "CHN": 2015, "COL": 2007, "CZE": 2003, "GRC": 2003, "HUN": 2003, "IND": 2004, "MYS": 2005,
    "MEX": 2003, "PHL": 2009, "POL": 2003, "RUS": 2004, "ZAF": 2003, "KOR": 2003, "THA": 2003,
}
ISO3_TO_ISO2 = {
    "AUS": "AU", "AUT": "AT", "BEL": "BE", "BRA": "BR", "CAN": "CA", "CHE": "CH", "CHN": "CN", "COL": "CO",
    "CZE": "CZ", "DEU": "DE", "DNK": "DK", "ESP": "ES", "FIN": "FI", "FRA": "FR", "GBR": "GB", "GRC": "GR",
    "HKG": "HK", "HUN": "HU", "IND": "IN", "ISR": "IL", "ITA": "IT", "JPN": "JP", "KOR": "KR", "MEX": "MX",
    "MYS": "MY", "NLD": "NL", "NOR": "NO", "NZL": "NZ", "PHL": "PH", "POL": "PL", "PRT": "PT", "RUS": "RU",
    "SGP": "SG", "SWE": "SE", "THA": "TH", "USA": "US", "ZAF": "ZA",
}
BIS_OBS_TO_MILLION = 1000.0
ASSET_TO_PIP = {"st": "ST_DEBT", "lt": "LT_DEBT", "eq": "EQUITY"}
ASSET_CODE_RESTATEMENT = {"ST_DEBT": "B", "LT_DEBT": "B", "EQUITY": "E"}
project_root = Path.cwd().resolve().parent
data_dir = project_root / "_data"
pip_dir = project_root / "data"
SHC_A7_PATH = project_root / "manual added data" / "shc_app07_2020.csv"
SHC_A8_PATH = project_root / "manual added data" / "shc_app08_2020.csv"
BIS_IDS_PATH = data_dir / "bis_ids_foreign_currency_q.parquet"
BIS_IDS_ALL_PATH = data_dir / "bis_ids_all_currency_q.parquet"
BIS_TOTAL_PATH = data_dir / "bis_total_debt_all_currency_q.parquet"
oecd = pd.read_parquet(data_dir / "oecd_t720.parquet")
fx = pd.read_parquet(data_dir / "oecd_implied_fx_xdc_per_usd.parquet")
bis = pd.read_parquet(data_dir / "bis_debt_securities_cleaned.parquet")
wb = pd.read_parquet(data_dir / "wb_data360_wdi_selected.parquet")
oecd_base = oecd[
    (oecd["frequency"].astype(str) == "A")
    & (oecd["sector"].astype(str) == "S1")
    & (oecd["consolidation"].astype(str) == "N")
    & (oecd["accounting_entry"].astype(str) == "L")
    & (oecd["transaction"].astype(str) == "LE")
    & (oecd["counterpart_area"].astype(str) == "W")
    & (oecd["counterpart_sector"].astype(str) == "S1")
    & (oecd["table_identifier"].astype(str) == "T0720")
].copy()
wb_eq = wb[wb["metric"].astype(str) == "market_cap_listed_domestic_companies_current_usd"].copy()
msci_path = data_dir / "data_msci_datastream.parquet"
if msci_path.exists():
    msci = pd.read_parquet(msci_path)
    msci_eq = msci[msci["metric"].astype(str) == "market equity outstanding"].copy()
    iso2_to_iso3 = {str(v): str(k) for k, v in ISO3_TO_ISO2.items()}
    msci_eq["ref_area"] = msci_eq["entity_id"].astype(str).map(iso2_to_iso3)
    msci_eq = msci_eq.dropna(subset=["ref_area", "obs_date", "value"]).copy()
    msci_eq["obs_date"] = pd.to_datetime(msci_eq["obs_date"], errors="coerce")
    msci_eq = msci_eq.dropna(subset=["obs_date"]).copy()
    msci_eq["time_period"] = msci_eq["obs_date"].dt.year.astype("Int64")
    msci_eq["value"] = pd.to_numeric(msci_eq["value"], errors="coerce")
    msci_eq = msci_eq.dropna(subset=["value", "time_period"]).copy()
    msci_eq = (
        msci_eq.sort_values(["ref_area", "time_period", "obs_date"])
        .groupby(["ref_area", "time_period"], as_index=False)
        .tail(1)
    )
else:
    msci_eq = pd.DataFrame(columns=["ref_area", "time_period", "value"])
_msci_splice_ratio_cache = {}
restatement_path = project_root / "manual added data" / "Restatement_Matrices.dta"
restatement = pd.read_stata(restatement_path)[[
    "Methodology", "Year", "Asset_Class_Code", "Destination", "Destination_Restated", "Value"
]].copy()
restatement["Year"] = pd.to_numeric(restatement["Year"], errors="coerce").astype("Int64")
restatement["Value"] = pd.to_numeric(restatement["Value"], errors="coerce")
_restatement_cache_method = {}
_RESTATEMENT_ENHANCED = {"NOR", "USA"}
_RESTATEMENT_FUND = {"AUT", "BEL", "FIN", "FRA", "DEU", "ITA", "NLD", "PRT", "ESP", "GRC", "AUS", "CAN", "DNK", "SWE", "CHE", "GBR", "NOR"}
def _restatement_method_for_issuer(issuer: str) -> str:
    iso = str(issuer)
    if iso in _RESTATEMENT_ENHANCED:
        return "Enhanced Fund Holdings"
    if iso in _RESTATEMENT_FUND:
        return "Fund Holdings"
    return "Issuance"
def _restatement_matrix_for_year_asset(year: int, asset_code: str, methodology: str) -> pd.DataFrame:
    key = (int(year), str(asset_code), str(methodology))
    if key in _restatement_cache_method:
        return _restatement_cache_method[key]
    x = restatement[
        (restatement["Asset_Class_Code"] == asset_code)
        & (restatement["Methodology"].astype(str) == str(methodology))
    ].dropna(subset=["Year", "Value"]).copy()
    if x.empty and str(methodology) != "Issuance":
        x = restatement[
            (restatement["Asset_Class_Code"] == asset_code)
            & (restatement["Methodology"].astype(str) == "Issuance")
        ].dropna(subset=["Year", "Value"]).copy()
    if x.empty:
        out = pd.DataFrame(columns=["Destination", "Destination_Restated", "Value"])
        _restatement_cache_method[key] = out
        return out
    x["dist"] = (x["Year"].astype(int) - int(year)).abs()
    y = int(x.sort_values(["dist", "Year"]).iloc[0]["Year"])
    yx = x[x["Year"].astype(int) == y][["Destination", "Destination_Restated", "Value"]].copy()
    _restatement_cache_method[key] = yx
    return yx
def _apply_restatement_by_issuer(agg_df: pd.DataFrame) -> pd.DataFrame:
    if agg_df.empty:
        return agg_df.copy()
    parts = []
    for (tp, ac), g in agg_df.groupby(["TIME_PERIOD", "asset_class"], dropna=False):
        year = int(str(tp))
        asset_code = ASSET_CODE_RESTATEMENT.get(str(ac), "B")
        gg = g.copy()
        gg["methodology"] = "Issuance"
        for method, gm in gg.groupby("methodology", dropna=False):
            mat = _restatement_matrix_for_year_asset(year, asset_code, str(method))
            if mat.empty:
                gm2 = gm.copy()
                gm2["issuer_restated"] = gm2["issuer"].astype(str)
                gm2["value_restated_usd"] = pd.to_numeric(gm2["value_usd"], errors="coerce")
                out = gm2.groupby(["issuer_restated", "TIME_PERIOD", "asset_class"], as_index=False)["value_restated_usd"].sum(min_count=1)
                out = out.rename(columns={"issuer_restated": "issuer", "value_restated_usd": "value_usd"})
                parts.append(out)
                continue
            m = gm.merge(mat, left_on="issuer", right_on="Destination", how="left")
            m["Destination_Restated"] = m["Destination_Restated"].fillna(m["issuer"])
            m["Value"] = m["Value"].fillna(1.0)
            m["value_restated_usd"] = pd.to_numeric(m["value_usd"], errors="coerce") * pd.to_numeric(m["Value"], errors="coerce")
            out = m.groupby(["Destination_Restated", "TIME_PERIOD", "asset_class"], as_index=False)["value_restated_usd"].sum(min_count=1)
            out = out.rename(columns={"Destination_Restated": "issuer", "value_restated_usd": "value_usd"})
            parts.append(out)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["issuer", "TIME_PERIOD", "asset_class", "value_usd"])
def _prepare_pip_bilateral_clean(file_name: str, value_col: str = "value_usd", issuer_col: str = "issuer") -> pd.DataFrame:
    x = pd.read_parquet(
        pip_dir / file_name,
        columns=[issuer_col, "COUNTRY", "TIME_PERIOD", "asset_class", value_col],
    )
    x = x.rename(columns={issuer_col: "issuer", value_col: "value_usd"})
    x = x[x["TIME_PERIOD"].astype(str) == str(TARGET_YEAR)].copy()
    if x.empty:
        return pd.DataFrame(columns=["issuer", "TIME_PERIOD", "asset_class", "value_usd"])
    # Appendix C: drop OFC investors to avoid double counting.
    x = x[~x["COUNTRY"].astype(str).isin(OFC_ISO3)].copy()
    if x.empty:
        return pd.DataFrame(columns=["issuer", "TIME_PERIOD", "asset_class", "value_usd"])
    # Keep only sample issuers/year needed by the final table.
    x = x[x["issuer"].astype(str).isin(COUNTRY_ORDER)].copy()
    x = x[~x["COUNTRY"].astype(str).isin(OFC_ISO3)].copy()
    if x.empty:
        return pd.DataFrame(columns=["issuer", "TIME_PERIOD", "asset_class", "value_usd"])
    x = x[x["COUNTRY"].astype(str) != x["issuer"].astype(str)].copy()
    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")
    _pos = x["value_usd"] > 0
    x.loc[_pos, "value_usd"] = np.ceil(x.loc[_pos, "value_usd"] / 1000.0) * 1000.0
    # Appendix C: round up positive restated holdings to reporting minimum ($1,000).
    _pos = x["value_usd"] > 0
    x.loc[_pos, "value_usd"] = np.ceil(x.loc[_pos, "value_usd"] / 1000.0) * 1000.0
    # Appendix C: keep confidential/small (<0.5m USD) holdings as missing.
    x.loc[(x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD)), "value_usd"] = pd.NA
    agg = x.groupby(["issuer", "TIME_PERIOD", "asset_class"], as_index=False)["value_usd"].sum(min_count=1)
    return agg
pip_foreign_agg = _prepare_pip_bilateral_clean("pip_bilateral_positions.parquet")
pip_foreign_local_agg = _prepare_pip_bilateral_clean("pip_local_foreign_allocated.parquet", value_col="value_local", issuer_col="COUNTERPART_COUNTRY")
pip_reserve_agg = _prepare_pip_bilateral_clean("pip_bilateral_positions_reserve.parquet")
USE_LOCAL_FOREIGN_DEBT = True
def _pip_sum_million(agg_df: pd.DataFrame, issuer: str, year: int, asset_class: str) -> Optional[float]:
    x = agg_df[
        (agg_df["issuer"].astype(str) == issuer)
        & (agg_df["TIME_PERIOD"].astype(str) == str(year))
        & (agg_df["asset_class"].astype(str) == asset_class)
    ]
    if x.empty:
        return None
    v = pd.to_numeric(x["value_usd"], errors="coerce").sum(min_count=1)
    return None if pd.isna(v) else float(v) / 1e6
def _domestic_share(total_million: Optional[float], foreign_million: Optional[float]) -> Optional[float]:
    if total_million is None or foreign_million is None:
        return None
    t = float(total_million)
    f = float(foreign_million)
    if pd.isna(t) or pd.isna(f) or t <= 0:
        return None
    # Keep raw ratio by instruction formula; do not clip negatives or values above one.
    return (t - f) / t
def _share_in_reserves(reserve_million: Optional[float], foreign_total_million: Optional[float]) -> Optional[float]:
    if reserve_million is None or foreign_total_million is None:
        return None
    r = float(reserve_million)
    f = float(foreign_total_million)
    if pd.isna(r) or pd.isna(f) or f <= 0:
        return None
    out = r / f
    return max(0.0, min(1.0, out))
# SHC naming differs from ISO3 labels for a few issuers.
ISO3_TO_SHC_NAME = {
    "CAN": "Canada", "USA": "United States", "AUT": "Austria", "BEL": "Belgium", "DNK": "Denmark",
    "FIN": "Finland", "FRA": "France", "DEU": "Germany", "ISR": "Israel", "ITA": "Italy",
    "NLD": "Netherlands", "NOR": "Norway", "PRT": "Portugal", "ESP": "Spain", "SWE": "Sweden",
    "CHE": "Switzerland", "GBR": "United Kingdom", "AUS": "Australia", "HKG": "Hong Kong",
    "JPN": "Japan", "NZL": "New Zealand", "SGP": "Singapore", "BRA": "Brazil",
    "CHN": "China, mainland (1)", "COL": "Colombia", "CZE": "Czech Republic", "GRC": "Greece",
    "HUN": "Hungary", "IND": "India", "MYS": "Malaysia", "MEX": "Mexico",
    "PHL": "Philippines", "POL": "Poland", "RUS": "Russia", "ZAF": "South Africa",
    "KOR": "Korea, South", "THA": "Thailand",
}
def _read_shc_country_total_million(path: Path, country_name: str) -> Optional[float]:
    if not path.exists():
        return None
    rows = []
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        for r in reader:
            rows.append(r)
    if not rows:
        return None
    header_idx = None
    for i, r in enumerate(rows):
        if len(r) > 0 and str(r[0]).strip() == "Country or region of issuer":
            header_idx = i
            break
    if header_idx is None:
        return None
    vals = []
    for r in rows[header_idx + 1 :]:
        if len(r) < 2:
            continue
        c = str(r[0]).strip()
        if c == "":
            continue
        if c.casefold() != str(country_name).casefold():
            continue
        v = pd.to_numeric(r[1], errors="coerce")
        if pd.notna(v):
            vals.append(float(v))
    if not vals:
        return None
    return float(sum(vals))
def _usa_bilateral_st_lt_million() -> pd.DataFrame:
    p = pip_dir / "pip_bilateral_positions.parquet"
    if not p.exists():
        return pd.DataFrame(columns=["issuer", "asset_class", "usa_mn"])
    x = pd.read_parquet(p, columns=["COUNTRY", "issuer", "TIME_PERIOD", "asset_class", "value_usd"])
    x = x[
        (x["COUNTRY"].astype(str) == "USA")
        & (x["TIME_PERIOD"].astype(str) == str(TARGET_YEAR))
        & (x["issuer"].astype(str).isin(COUNTRY_ORDER))
        & (x["asset_class"].astype(str).isin(["ST_DEBT", "LT_DEBT"]))
    ].copy()
    if x.empty:
        return pd.DataFrame(columns=["issuer", "asset_class", "usa_mn"])
    x["value_usd"] = pd.to_numeric(x["value_usd"], errors="coerce")
    x.loc[(x["value_usd"] > 0) & (x["value_usd"] < float(SMALL_HOLDING_USD)), "value_usd"] = pd.NA
    g = x.groupby(["issuer", "asset_class"], as_index=False)["value_usd"].sum(min_count=1)
    g["usa_mn"] = pd.to_numeric(g["value_usd"], errors="coerce") / 1e6
    return g[["issuer", "asset_class", "usa_mn"]]
# --- OECD helpers ---
def _oecd_unit_for_country_year(country_iso3: str, year: int) -> Optional[str]:
    x = oecd_base[(oecd_base["reference_area"] == country_iso3) & (oecd_base["time_period"].astype(str) == str(year))]
    if x.empty:
        return None
    if (x["unit_measure"] == "XDC").any():
        return "XDC"
    if (x["unit_measure"] == "USD").any():
        return "USD"
    return None
def _to_usd_million_oecd(country_iso3: str, year: int, raw_million: float, unit_measure: str) -> Optional[float]:
    if unit_measure == "USD":
        return float(raw_million)
    if unit_measure == "XDC":
        row = fx[(fx["reference_area"] == country_iso3) & (fx["time_period"].astype(str) == str(year))]
        if row.empty:
            return None
        f = float(row["fx_xdc_per_usd"].iloc[0])
        if f <= 0:
            return None
        return float(raw_million) / f
    return None
def _nearest_year_ratio_for_debt_split_oecd(country_iso3: str, target_year: int, debt_ccy: str) -> Optional[float]:
    x_all = oecd_base[(oecd_base["reference_area"] == country_iso3) & (oecd_base["currency_denom"] == debt_ccy)]
    if x_all.empty:
        return None
    years = sorted(x_all["time_period"].dropna().astype(str).unique().tolist())
    candidates = []
    for y in years:
        x = x_all[x_all["time_period"].astype(str) == y]
        st = float(x[(x["financial_instrument"] == "F3") & (x["original_maturity"] == "S")]["value"].sum())
        lt = float(x[(x["financial_instrument"] == "F3") & (x["original_maturity"] == "L")]["value"].sum())
        if st > 0 and lt > 0:
            candidates.append((int(y), st / (st + lt)))
    if not candidates:
        return None
    return min(candidates, key=lambda t: abs(t[0] - int(target_year)))[1]
def _nearest_year_ratio_for_equity_split_oecd(country_iso3: str, target_year: int) -> Optional[float]:
    x_all = oecd_base[oecd_base["reference_area"] == country_iso3]
    if x_all.empty:
        return None
    years = sorted(x_all["time_period"].dropna().astype(str).unique().tolist())
    candidates = []
    for y in years:
        x = x_all[x_all["time_period"].astype(str) == y]
        f51 = float(x[(x["financial_instrument"] == "F51") & (x["original_maturity"].isin(["_Z", "T"]))]["value"].sum())
        f519 = float(x[(x["financial_instrument"] == "F519") & (x["original_maturity"].isin(["_Z", "T"]))]["value"].sum())
        ce = max(0.0, f51 - f519)
        if ce <= 0 and f51 > 0:
            ce = f51
        f5 = float(x[(x["financial_instrument"] == "F5") & (x["original_maturity"].isin(["_Z", "T"]))]["value"].sum())
        if ce > 0 and f5 > 0:
            candidates.append((int(y), ce / f5))
    if not candidates:
        return None
    return min(candidates, key=lambda t: abs(t[0] - int(target_year)))[1]
def _oecd_debt_st_lt_usd_million(country_iso3: str, year: int) -> tuple[Optional[float], Optional[float]]:
    u = _oecd_unit_for_country_year(country_iso3, year)
    if u is None:
        return None, None
    x = oecd_base[
        (oecd_base["reference_area"] == country_iso3)
        & (oecd_base["time_period"].astype(str) == str(year))
        & (oecd_base["unit_measure"] == u)
    ]
    if x.empty:
        return None, None
    debt_ccy = next((ccy for ccy in ["_T", "XDC", "USD"] if (x["currency_denom"] == ccy).any()), None)
    if debt_ccy is None:
        return None, None
    st_raw = float(x[(x["financial_instrument"] == "F3") & (x["original_maturity"] == "S") & (x["currency_denom"] == debt_ccy)]["value"].sum())
    lt_raw = float(x[(x["financial_instrument"] == "F3") & (x["original_maturity"] == "L") & (x["currency_denom"] == debt_ccy)]["value"].sum())
    tot_raw = float(x[(x["financial_instrument"] == "F3") & (x["original_maturity"].isin(["T", "_Z"])) & (x["currency_denom"] == debt_ccy)]["value"].sum())
    if (st_raw == 0.0 or lt_raw == 0.0) and tot_raw > 0.0:
        r = _nearest_year_ratio_for_debt_split_oecd(country_iso3, int(year), debt_ccy)
        if r is not None:
            st_raw = tot_raw * r
            lt_raw = tot_raw * (1.0 - r)
    return _to_usd_million_oecd(country_iso3, year, st_raw, u), _to_usd_million_oecd(country_iso3, year, lt_raw, u)
def _oecd_equity_usd_million(country_iso3: str, year: int) -> Optional[float]:
    u = _oecd_unit_for_country_year(country_iso3, year)
    if u is None:
        return None
    x = oecd_base[
        (oecd_base["reference_area"] == country_iso3)
        & (oecd_base["time_period"].astype(str) == str(year))
        & (oecd_base["unit_measure"] == u)
    ]
    if x.empty:
        return None
    f51 = float(x[(x["financial_instrument"] == "F51") & (x["original_maturity"].isin(["_Z", "T"]))]["value"].sum())
    f519 = float(x[(x["financial_instrument"] == "F519") & (x["original_maturity"].isin(["_Z", "T"]))]["value"].sum())
    ce_raw = max(0.0, f51 - f519)
    if ce_raw <= 0.0 and f51 > 0.0:
        ce_raw = f51
    f5_raw = float(x[(x["financial_instrument"] == "F5") & (x["original_maturity"].isin(["_Z", "T"]))]["value"].sum())
    if ce_raw <= 0.0 and f5_raw > 0.0:
        r = _nearest_year_ratio_for_equity_split_oecd(country_iso3, int(year))
        ce_raw = f5_raw * r if r is not None else f5_raw
    return _to_usd_million_oecd(country_iso3, year, ce_raw, u)
# --- BIS helper ---
def _bis_base(country_iso2: str) -> pd.DataFrame:
    x = bis[
        (bis["REF_AREA"].astype(str) == country_iso2)
        & (bis["COUNTERPART_SECTOR"].astype(str) == "S1")
        & (bis["CONSOLIDATION"].astype(str) == "N")
        & (bis["ACCOUNTING_ENTRY"].astype(str) == "L")
        & (bis["STO"].astype(str) == "LE")
        & (bis["INSTR_ASSET"].astype(str) == "F3")
        & (bis["TRANSFORMATION"].astype(str) == "N")
        & (bis["UNIT_MEASURE"].astype(str) == "USD")
        & (bis["PRICES"].astype(str) == "V")
        & (bis["EXPENDITURE"].astype(str) == "_Z")
        & (bis["REF_SECTOR"].astype(str).isin(["S11", "S12", "S13"]))
    ].copy()
    x["OBS_VALUE"] = pd.to_numeric(x["OBS_VALUE"], errors="coerce")
    return x
def _bis_pick_currency_slice(x: pd.DataFrame) -> pd.DataFrame:
    if x.empty:
        return x
    present = set(x["CURRENCY_DENOM"].astype(str).unique().tolist())
    # Prefer total first, then local-currency coded totals in this dataset.
    for c in ["_T", "XDC", "X1", "USD"]:
        if c in present:
            return x[x["CURRENCY_DENOM"].astype(str) == c].copy()
    return x
def _bis_pick_valuation_slice(x: pd.DataFrame) -> pd.DataFrame:
    if x.empty:
        return x
    vals = set(x["VALUATION"].astype(str).unique().tolist())
    for v in ["M", "N", "F"]:
        if v in vals:
            return x[x["VALUATION"].astype(str) == v].copy()
    return x
def _bis_nearest_st_ratio(country_iso2: str, target_year: int, currency_denom: Optional[str] = None) -> Optional[float]:
    q = _bis_base(country_iso2)
    q = q[(q["FREQ"].astype(str) == "Q") & (q["TIME_PERIOD"].astype(str).str.endswith("-Q4"))].copy()
    if currency_denom is not None:
        q = q[q["CURRENCY_DENOM"].astype(str) == str(currency_denom)].copy()
    else:
        q = _bis_pick_currency_slice(q)
    q = _bis_pick_valuation_slice(q)
    if q.empty:
        return None
    out = []
    for tp, g in q.groupby("TIME_PERIOD"):
        st = float(g[g["MATURITY"].astype(str) == "S"]["OBS_VALUE"].sum())
        lt = float(g[g["MATURITY"].astype(str) == "L"]["OBS_VALUE"].sum())
        if st > 0 and lt > 0:
            out.append((int(str(tp)[:4]), st / (st + lt)))
    if not out:
        return None
    return min(out, key=lambda t: abs(t[0] - int(target_year)))[1]
def _bis_st_share_from_available_issuers(country_iso2: str, period: str, freq: str, currency_denom: Optional[str] = None) -> Optional[float]:
    x = _bis_base(country_iso2)
    x = x[(x["FREQ"].astype(str) == str(freq)) & (x["TIME_PERIOD"].astype(str) == str(period))].copy()
    if currency_denom is not None:
        x = x[x["CURRENCY_DENOM"].astype(str) == str(currency_denom)].copy()
    else:
        x = _bis_pick_currency_slice(x)
    x = _bis_pick_valuation_slice(x)
    if x.empty:
        return None
    st_sum = 0.0
    lt_sum = 0.0
    for _, g in x.groupby("REF_SECTOR", dropna=False):
        st_i = float(g[g["MATURITY"].astype(str) == "S"]["OBS_VALUE"].sum())
        lt_i = float(g[g["MATURITY"].astype(str) == "L"]["OBS_VALUE"].sum())
        if st_i > 0.0 and lt_i > 0.0:
            st_sum += st_i
            lt_sum += lt_i
    den = st_sum + lt_sum
    if den <= 0.0:
        return None
    return st_sum / den
def _bis_global_st_ratio(period: str, freq: str, currency_denom: Optional[str] = None) -> Optional[float]:
    x = bis[
        (bis["COUNTERPART_SECTOR"].astype(str) == "S1")
        & (bis["CONSOLIDATION"].astype(str) == "N")
        & (bis["ACCOUNTING_ENTRY"].astype(str) == "L")
        & (bis["STO"].astype(str) == "LE")
        & (bis["INSTR_ASSET"].astype(str) == "F3")
        & (bis["TRANSFORMATION"].astype(str) == "N")
        & (bis["UNIT_MEASURE"].astype(str) == "USD")
        & (bis["PRICES"].astype(str) == "V")
        & (bis["EXPENDITURE"].astype(str) == "_Z")
        & (bis["REF_SECTOR"].astype(str).isin(["S11", "S12", "S13"]))
        & (bis["FREQ"].astype(str) == str(freq))
        & (bis["TIME_PERIOD"].astype(str) == str(period))
    ].copy()
    if x.empty:
        return None
    if currency_denom is not None:
        x = x[x["CURRENCY_DENOM"].astype(str) == str(currency_denom)].copy()
    x = _bis_pick_valuation_slice(x)
    if x.empty:
        return None
    x["OBS_VALUE"] = pd.to_numeric(x["OBS_VALUE"], errors="coerce")
    st = float(x[x["MATURITY"].astype(str) == "S"]["OBS_VALUE"].sum())
    lt = float(x[x["MATURITY"].astype(str) == "L"]["OBS_VALUE"].sum())
    den = st + lt
    if den <= 0.0:
        return None
    return st / den
def _bis_domestic_st_lt_usd_million(country_iso2: str, year: int) -> tuple[Optional[float], Optional[float]]:
    q = _bis_base(country_iso2)
    periods = [("Q", f"{int(year)}-Q4"), ("A", str(int(year)))]
    for freq, period in periods:
        x = q[(q["FREQ"].astype(str) == str(freq)) & (q["TIME_PERIOD"].astype(str) == str(period))].copy()
        if x.empty:
            continue
        x = _bis_pick_currency_slice(x)
        x = _bis_pick_valuation_slice(x)
        if x.empty:
            continue
        ccy = str(x["CURRENCY_DENOM"].astype(str).iloc[0]) if "CURRENCY_DENOM" in x.columns else None
        r_issuer = _bis_st_share_from_available_issuers(country_iso2, str(period), str(freq), currency_denom=ccy)
        r_nearest = _bis_nearest_st_ratio(country_iso2, int(year), currency_denom=ccy)
        r_global = _bis_global_st_ratio(str(period), str(freq), currency_denom=ccy)
        st_obs = float(x[x["MATURITY"].astype(str) == "S"]["OBS_VALUE"].sum())
        lt_obs = float(x[x["MATURITY"].astype(str) == "L"]["OBS_VALUE"].sum())
        total_obs = float(x[x["MATURITY"].astype(str).isin(["T", "TT", "LS"])]["OBS_VALUE"].sum())
        st_sum = 0.0
        lt_sum = 0.0
        for _, g in x.groupby("REF_SECTOR", dropna=False):
            st_i = float(g[g["MATURITY"].astype(str) == "S"]["OBS_VALUE"].sum())
            lt_i = float(g[g["MATURITY"].astype(str) == "L"]["OBS_VALUE"].sum())
            tot_i = float(g[g["MATURITY"].astype(str).isin(["T", "TT", "LS"])]["OBS_VALUE"].sum())
            if st_i > 0.0 and lt_i > 0.0:
                st_sum += st_i
                lt_sum += lt_i
                continue
            if tot_i > 0.0:
                rr = r_issuer if r_issuer is not None else (r_nearest if r_nearest is not None else r_global)
                if rr is not None:
                    st_sum += tot_i * float(rr)
                    lt_sum += tot_i * (1.0 - float(rr))
                    continue
            st_sum += st_i
            lt_sum += lt_i
        if (st_sum <= 0.0 or lt_sum <= 0.0) and total_obs > 0.0:
            rr = r_nearest if r_nearest is not None else (r_issuer if r_issuer is not None else r_global)
            if rr is not None:
                st_sum = total_obs * float(rr)
                lt_sum = total_obs * (1.0 - float(rr))
        if st_sum > 0.0 or lt_sum > 0.0:
            return float(st_sum) * float(BIS_OBS_TO_MILLION), float(lt_sum) * float(BIS_OBS_TO_MILLION)
    return None, None
def _bis_has_raw_st_lt_split(country_iso2: str, year: int) -> bool:
    q = _bis_base(country_iso2)
    periods = [("Q", f"{int(year)}-Q4"), ("A", str(int(year)))]
    for freq, period in periods:
        x = q[(q["FREQ"].astype(str) == str(freq)) & (q["TIME_PERIOD"].astype(str) == str(period))].copy()
        if x.empty:
            continue
        x = _bis_pick_currency_slice(x)
        x = _bis_pick_valuation_slice(x)
        if x.empty:
            continue
        st_obs = float(x[x["MATURITY"].astype(str) == "S"]["OBS_VALUE"].sum())
        lt_obs = float(x[x["MATURITY"].astype(str) == "L"]["OBS_VALUE"].sum())
        if st_obs > 0.0 and lt_obs > 0.0:
            return True
    return False
def _bis_ids_foreign_st_lt_usd_million(country_iso2: str, year: int) -> tuple[float, float]:
    if not BIS_IDS_PATH.exists():
        return 0.0, 0.0
    ids = pd.read_parquet(BIS_IDS_PATH).copy()
    for c in [
        "MEASURE", "MARKET", "ISSUE_TYPE", "ISSUE_CUR", "ISSUE_RE_MAT", "ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL",
        "ISSUER_BUS_IMM", "ISSUER_BUS_ULT", "ISSUE_CUR_GROUP", "ISSUE_OR_MAT", "TIME_PERIOD", "ISSUER_RES",
    ]:
        if c in ids.columns:
            ids[c] = ids[c].astype(str)
    ids = ids[
        (ids["ISSUER_RES"] == str(country_iso2))
        & (ids["MEASURE"] == "I")
        & (ids["ISSUE_CUR_GROUP"] == "F")
        & (ids["TIME_PERIOD"] == f"{year}-Q4")
    ].copy()
    if ids.empty:
        return 0.0, 0.0
    if "MARKET" in ids.columns and (ids["MARKET"].astype(str) == "C").any():
        ids = ids[ids["MARKET"].astype(str) == "C"].copy()
    if "ISSUE_TYPE" in ids.columns and (ids["ISSUE_TYPE"].astype(str) == "A").any():
        ids = ids[ids["ISSUE_TYPE"].astype(str) == "A"].copy()
    if "ISSUE_CUR" in ids.columns and (ids["ISSUE_CUR"].astype(str) == "TO1").any():
        ids = ids[ids["ISSUE_CUR"].astype(str) == "TO1"].copy()
    if "ISSUE_RE_MAT" in ids.columns and (ids["ISSUE_RE_MAT"].astype(str) == "A").any():
        ids = ids[ids["ISSUE_RE_MAT"].astype(str) == "A"].copy()
    for _c in ["ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL"]:
        if _c in ids.columns and (ids[_c].astype(str) == "A").any():
            ids = ids[ids[_c].astype(str) == "A"].copy()
    if "ISSUER_BUS_IMM" in ids.columns:
        ids_agg = ids[ids["ISSUER_BUS_IMM"].astype(str) == "1"].copy()
        if not ids_agg.empty:
            ids = ids_agg
    if "million_usd" in ids.columns:
        ids["million_usd"] = pd.to_numeric(ids["million_usd"], errors="coerce")
    else:
        ids["million_usd"] = (
            pd.to_numeric(ids.get("OBS_VALUE"), errors="coerce")
            * (10 ** pd.to_numeric(ids.get("UNIT_MULT"), errors="coerce").fillna(0))
        ) / 1e6
    st = float(ids[ids["ISSUE_OR_MAT"] == "C"]["million_usd"].sum())
    lt = float(ids[ids["ISSUE_OR_MAT"] == "K"]["million_usd"].sum())
    return st, lt
def _bis_ids_all_st_lt_usd_million(country_iso2: str, year: int) -> tuple[float, float]:
    # Preferred source: all-currency IDS pull (A/D/F groups).
    p = BIS_IDS_ALL_PATH if 'BIS_IDS_ALL_PATH' in globals() else None
    if p is not None and Path(p).exists():
        ids = pd.read_parquet(p).copy()
        for c in ["ISSUER_RES", "ISSUE_OR_MAT", "TIME_PERIOD", "ISSUE_CUR_GROUP", "ISSUER_BUS_IMM"]:
            if c in ids.columns:
                ids[c] = ids[c].astype(str)
        ids = ids[
            (ids.get("ISSUER_RES", "").astype(str) == str(country_iso2))
            & (ids.get("TIME_PERIOD", "").astype(str) == f"{year}-Q4")
            & (ids.get("ISSUE_OR_MAT", "").astype(str).isin(["A", "C", "K"]))
            & (ids.get("ISSUE_CUR_GROUP", "").astype(str) == "A")
            & (ids.get("MEASURE", "").astype(str) == "I")
        ].copy()
        if not ids.empty:
            if "MARKET" in ids.columns and (ids["MARKET"].astype(str) == "C").any():
                ids = ids[ids["MARKET"].astype(str) == "C"].copy()
            if "ISSUE_TYPE" in ids.columns and (ids["ISSUE_TYPE"].astype(str) == "A").any():
                ids = ids[ids["ISSUE_TYPE"].astype(str) == "A"].copy()
            if "ISSUE_CUR" in ids.columns and (ids["ISSUE_CUR"].astype(str) == "TO1").any():
                ids = ids[ids["ISSUE_CUR"].astype(str) == "TO1"].copy()
            if "ISSUE_RE_MAT" in ids.columns and (ids["ISSUE_RE_MAT"].astype(str) == "A").any():
                ids = ids[ids["ISSUE_RE_MAT"].astype(str) == "A"].copy()
            for _c in ["ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL"]:
                if _c in ids.columns and (ids[_c].astype(str) == "A").any():
                    ids = ids[ids[_c].astype(str) == "A"].copy()
            if "ISSUER_BUS_IMM" in ids.columns:
                ids_agg = ids[ids["ISSUER_BUS_IMM"].astype(str) == "1"].copy()
                if not ids_agg.empty:
                    ids = ids_agg
            if "million_usd" in ids.columns:
                ids["million_usd"] = pd.to_numeric(ids["million_usd"], errors="coerce")
            else:
                ids["million_usd"] = (
                    pd.to_numeric(ids.get("OBS_VALUE"), errors="coerce")
                    * (10 ** pd.to_numeric(ids.get("UNIT_MULT"), errors="coerce").fillna(0))
                ) / 1e6
            st = float(ids[ids["ISSUE_OR_MAT"] == "C"]["million_usd"].sum())
            lt = float(ids[ids["ISSUE_OR_MAT"] == "K"]["million_usd"].sum())
            tot = float(ids[ids["ISSUE_OR_MAT"] == "A"]["million_usd"].sum())
            if tot > 0.0 and (st <= 0.0 or lt <= 0.0):
                r = _bis_total_nearest_st_ratio(country_iso2, int(year))
                if r is None:
                    r = _bis_global_st_ratio(f"{int(year)}-Q4", "Q", currency_denom=None)
                if r is not None:
                    st = tot * float(r)
                    lt = tot * (1.0 - float(r))
                else:
                    if st > 0.0 and lt <= 0.0:
                        lt = max(0.0, tot - st)
                    elif lt > 0.0 and st <= 0.0:
                        st = max(0.0, tot - lt)
            return st, lt
    return _bis_ids_foreign_st_lt_usd_million(country_iso2, year)
def _bis_total_nearest_st_ratio(country_iso2: str, target_year: int) -> Optional[float]:
    if not BIS_IDS_ALL_PATH.exists():
        return None
    ids = pd.read_parquet(BIS_IDS_ALL_PATH).copy()
    for c in ["ISSUER_RES", "ISSUE_OR_MAT", "TIME_PERIOD", "ISSUE_CUR_GROUP", "MEASURE", "MARKET", "ISSUE_TYPE", "ISSUE_CUR", "ISSUE_RE_MAT", "ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL", "ISSUER_BUS_IMM"]:
        if c in ids.columns:
            ids[c] = ids[c].astype(str)
    ids = ids[
        (ids.get("ISSUER_RES", "").astype(str) == str(country_iso2))
        & (ids.get("ISSUE_CUR_GROUP", "").astype(str) == "A")
        & (ids.get("ISSUE_OR_MAT", "").astype(str).isin(["C", "K"]))
        & (ids.get("MEASURE", "").astype(str) == "I")
    ].copy()
    if ids.empty:
        return None
    if "MARKET" in ids.columns and (ids["MARKET"].astype(str) == "C").any():
        ids = ids[ids["MARKET"].astype(str) == "C"].copy()
    if "ISSUE_TYPE" in ids.columns and (ids["ISSUE_TYPE"].astype(str) == "A").any():
        ids = ids[ids["ISSUE_TYPE"].astype(str) == "A"].copy()
    if "ISSUE_CUR" in ids.columns and (ids["ISSUE_CUR"].astype(str) == "TO1").any():
        ids = ids[ids["ISSUE_CUR"].astype(str) == "TO1"].copy()
    if "ISSUE_RE_MAT" in ids.columns and (ids["ISSUE_RE_MAT"].astype(str) == "A").any():
        ids = ids[ids["ISSUE_RE_MAT"].astype(str) == "A"].copy()
    for _c in ["ISSUE_RATE", "ISSUE_RISK", "ISSUE_COL"]:
        if _c in ids.columns and (ids[_c].astype(str) == "A").any():
            ids = ids[ids[_c].astype(str) == "A"].copy()
    if "ISSUER_BUS_IMM" in ids.columns:
        ids_agg = ids[ids["ISSUER_BUS_IMM"].astype(str) == "1"].copy()
        if not ids_agg.empty:
            ids = ids_agg
    if "million_usd" in ids.columns:
        ids["million_usd"] = pd.to_numeric(ids["million_usd"], errors="coerce")
    else:
        ids["million_usd"] = (
            pd.to_numeric(ids.get("OBS_VALUE"), errors="coerce")
            * (10 ** pd.to_numeric(ids.get("UNIT_MULT"), errors="coerce").fillna(0))
        ) / 1e6
    cand = []
    for tp, g in ids.groupby("TIME_PERIOD", dropna=False):
        y = int(str(tp)[:4])
        st = float(g[g["ISSUE_OR_MAT"].astype(str) == "C"]["million_usd"].sum())
        lt = float(g[g["ISSUE_OR_MAT"].astype(str) == "K"]["million_usd"].sum())
        if st > 0.0 and lt > 0.0:
            cand.append((abs(y - int(target_year)), y, st / (st + lt)))
    if not cand:
        return None
    cand.sort(key=lambda t: (t[0], t[1]))
    return float(cand[0][2])
def _bis_total_st_lt_usd_million(country_iso2: str, year: int) -> tuple[float, float]:
    if not BIS_TOTAL_PATH.exists():
        return 0.0, 0.0
    tot = pd.read_parquet(BIS_TOTAL_PATH).copy()
    for c in ["ISSUER_RES", "TIME_PERIOD", "COUNTERPART_AREA", "REF_SECTOR", "ACCOUNTING_ENTRY", "STO", "INSTR_ASSET", "MATURITY", "CURRENCY_DENOM", "CUST_BREAKDOWN"]:
        if c in tot.columns:
            tot[c] = tot[c].astype(str)
    tot = tot[(tot.get("ISSUER_RES", "").astype(str) == str(country_iso2))].copy()
    tot_filters = {
        "ACCOUNTING_ENTRY": "L",
        "STO": "LE",
        "INSTR_ASSET": "F3",
        "CUST_BREAKDOWN": "_T",
    }
    for c, v in tot_filters.items():
        if c in tot.columns:
            tot = tot[tot[c].astype(str) == v].copy()
    if tot.empty:
        return 0.0, 0.0
    if "MATURITY" in tot.columns:
        for _m in ["T", "TT", "LS", "L", "S"]:
            _sel = tot[tot["MATURITY"].astype(str) == _m].copy()
            if not _sel.empty:
                tot = _sel
                break
    if "CURRENCY_DENOM" in tot.columns:
        for _c in ["_T", "XDC", "USD", "X1"]:
            _sel = tot[tot["CURRENCY_DENOM"].astype(str) == _c].copy()
            if not _sel.empty:
                tot = _sel
                break
    if "million_usd" in tot.columns:
        tot["million_usd"] = pd.to_numeric(tot["million_usd"], errors="coerce")
    else:
        tot["million_usd"] = (
            pd.to_numeric(tot.get("OBS_VALUE"), errors="coerce")
            * (10 ** pd.to_numeric(tot.get("UNIT_MULT"), errors="coerce").fillna(0))
        ) / 1e6
    _tp = tot.get("TIME_PERIOD", pd.Series(index=tot.index, dtype=object)).astype(str)
    _target = tot[_tp.isin([f"{year}-Q4", str(year)])].copy()
    if _target.empty:
        _yr = pd.to_numeric(_tp.str.slice(0, 4), errors="coerce")
        _ok = _yr.notna()
        if _ok.any():
            _nearest = int(_yr[_ok].astype(int).drop_duplicates().sort_values(key=lambda s: (s - int(year)).abs()).iloc[0])
            _target = tot[_yr == _nearest].copy()
    if _target.empty:
        return 0.0, 0.0
    tot = _target
    if "COUNTERPART_AREA" in tot.columns:
        _tot_xw = tot[tot["COUNTERPART_AREA"].astype(str) == "XW"].copy()
        if not _tot_xw.empty:
            tot = _tot_xw
        else:
            _cp_sum = (
                tot.groupby("COUNTERPART_AREA", as_index=False)["million_usd"]
                .sum(min_count=1)
                .sort_values("million_usd", ascending=False)
            )
            if not _cp_sum.empty and pd.notna(_cp_sum.iloc[0]["million_usd"]):
                _best_cp = str(_cp_sum.iloc[0]["COUNTERPART_AREA"])
                tot = tot[tot["COUNTERPART_AREA"].astype(str) == _best_cp].copy()
    if "REF_SECTOR" in tot.columns:
        _rs = tot["REF_SECTOR"].astype(str)
        _s_total = float(tot.loc[_rs.isin({"S1", "S1M", "S1ZS"}), "million_usd"].sum())
        _comp_total = float(tot.loc[_rs.isin({"S11", "S12", "S13"}), "million_usd"].sum())
        if _s_total > 0.0 and _comp_total > 0.0:
            total_obs = max(_s_total, _comp_total)
        elif _s_total > 0.0:
            total_obs = _s_total
        elif _comp_total > 0.0:
            total_obs = _comp_total
        else:
            total_obs = float(tot["million_usd"].sum())
    else:
        total_obs = float(tot["million_usd"].sum())
    if total_obs <= 0.0:
        return 0.0, 0.0
    r = _bis_total_nearest_st_ratio(country_iso2, int(year))
    if r is None:
        ids_st_mn, ids_lt_mn = _bis_ids_all_st_lt_usd_million(country_iso2, int(year))
        ids_tot = float(ids_st_mn) + float(ids_lt_mn)
        if ids_st_mn > 0.0 and ids_lt_mn > 0.0 and ids_tot > 0.0:
            r = float(ids_st_mn) / ids_tot
    if r is None:
        r = _bis_global_st_ratio(f"{year}-Q4", "Q", currency_denom=None)
    if r is None:
        return 0.0, 0.0
    st = total_obs * float(r)
    lt = total_obs * (1.0 - float(r))
    return st, lt
def _bis_debt_st_lt_usd_million(country_iso3: str, year: int) -> tuple[Optional[float], Optional[float]]:
    iso2 = ISO3_TO_ISO2.get(country_iso3)
    if iso2 is None:
        return None, None
    # Appendix C: total debt = domestic debt securities + international debt securities.
    ids_st_mn, ids_lt_mn = _bis_ids_all_st_lt_usd_million(iso2, int(year))
    dom_st_mn, dom_lt_mn = _bis_domestic_st_lt_usd_million(iso2, int(year))
    tot_st_mn, tot_lt_mn = _bis_total_st_lt_usd_million(iso2, int(year))
    tot_all = float(tot_st_mn) + float(tot_lt_mn)
    ids_all = float(ids_st_mn) + float(ids_lt_mn)
    dom_all_obs = float(dom_st_mn or 0.0) + float(dom_lt_mn or 0.0)
    dom_raw_split_ok = _bis_has_raw_st_lt_split(iso2, int(year))
    dom_equals_total = (tot_all > 0.0) and (abs(dom_all_obs - tot_all) <= 1e-6 * max(1.0, abs(tot_all)))
    # More precise missing/quality detection for BIS domestic leg:
    # 1) true missing/non-positive domestic observations
    dom_missing_basic = (dom_st_mn is None or dom_lt_mn is None) or (dom_all_obs <= 0.0)
    # 2) split-quality failure (missing raw split in source)
    dom_split_unreliable = (not dom_raw_split_ok)
    # 3) consistency failure versus total-minus-IDS identity
    dom_recon_all_raw = max(0.0, tot_all - ids_all) if tot_all > 0.0 else 0.0
    # tolerance is the larger of 2% relative or 100mn absolute to avoid noisy false positives
    _dom_tol = max(0.02 * max(1.0, abs(dom_recon_all_raw)), 100.0)
    dom_inconsistent_with_identity = (tot_all > 0.0) and (abs(dom_all_obs - dom_recon_all_raw) > _dom_tol)
    # 4) implausible scale: domestic exceeds total debt by meaningful margin
    dom_overshoots_total = (tot_all > 0.0) and (dom_all_obs > (tot_all + _dom_tol))
    use_reconstructed_domestic = (
        dom_missing_basic
        or dom_split_unreliable
        or dom_equals_total
        or dom_inconsistent_with_identity
        or dom_overshoots_total
    )
    # Appendix C rule: impute domestic when missing or quality checks fail.
    if use_reconstructed_domestic and tot_all > 0.0:
        dom_recon_all = dom_recon_all_raw
        if dom_recon_all <= 0.0:
            dom_st_mn = 0.0
            dom_lt_mn = 0.0
        else:
            rr = None
            if ids_all > 0.0 and float(ids_st_mn) > 0.0 and float(ids_lt_mn) > 0.0:
                rr = float(ids_st_mn) / ids_all
            if rr is None and dom_all_obs > 0.0 and (dom_st_mn is not None) and (dom_lt_mn is not None) and float(dom_st_mn) > 0.0 and float(dom_lt_mn) > 0.0:
                rr = float(dom_st_mn) / dom_all_obs
            if rr is None:
                rr = _bis_total_nearest_st_ratio(iso2, int(year))
            if rr is None:
                rr = _bis_global_st_ratio(f"{int(year)}-Q4", "Q", currency_denom=None)
            if rr is not None:
                dom_st_mn = dom_recon_all * float(rr)
                dom_lt_mn = dom_recon_all * (1.0 - float(rr))
            else:
                dom_st_mn = dom_recon_all
                dom_lt_mn = 0.0
    st_mn = float(ids_st_mn) + float(dom_st_mn or 0.0)
    lt_mn = float(ids_lt_mn) + float(dom_lt_mn or 0.0)
    if st_mn <= 0.0 and lt_mn <= 0.0:
        return None, None
    return float(st_mn), float(lt_mn)
def _wb_equity_usd_million(country_iso3: str, year: int) -> Optional[float]:
    x = wb_eq[(wb_eq["ref_area"] == country_iso3) & (wb_eq["time_period"].astype(str) == str(year))].copy()
    if x.empty:
        return None
    v = pd.to_numeric(x["value"], errors="coerce").sum(min_count=1)
    return None if pd.isna(v) else float(v) / 1e6
def _msci_equity_raw(country_iso3: str, year: int) -> Optional[float]:
    x = msci_eq[(msci_eq["ref_area"] == country_iso3) & (msci_eq["time_period"].astype(str) == str(year))]
    if x.empty:
        return None
    v = pd.to_numeric(x["value"], errors="coerce").sum(min_count=1)
    return None if pd.isna(v) else float(v)
def _msci_splice_ratio(country_iso3: str, year: int, anchor_name: str, anchor_getter) -> Optional[float]:
    key = (country_iso3, int(year), str(anchor_name))
    if key in _msci_splice_ratio_cache:
        return _msci_splice_ratio_cache[key]
    overlaps = []
    years = sorted({int(y) for y in pd.to_numeric(msci_eq[msci_eq["ref_area"] == country_iso3]["time_period"], errors="coerce").dropna().astype(int).tolist()})
    for yy in years:
        m = _msci_equity_raw(country_iso3, yy)
        a = anchor_getter(int(yy))
        if m is None or a is None:
            continue
        if float(m) <= 0.0 or float(a) <= 0.0:
            continue
        overlaps.append((abs(int(yy) - int(year)), int(yy), float(a) / float(m)))
    if not overlaps:
        _msci_splice_ratio_cache[key] = None
        return None
    overlaps.sort(key=lambda t: (t[0], t[1]))
    ratio = float(overlaps[0][2])
    _msci_splice_ratio_cache[key] = ratio
    return ratio
def _msci_equity_spliced_usd_million(country_iso3: str, year: int, anchor_name: str, anchor_getter) -> Optional[float]:
    m = _msci_equity_raw(country_iso3, year)
    if m is None or float(m) <= 0.0:
        return None
    ratio = _msci_splice_ratio(country_iso3, year, anchor_name, anchor_getter)
    if ratio is None or float(ratio) <= 0.0:
        return None
    return float(m) * float(ratio)
# Countries with instruction-level debt source switches over time.
SWITCH_COUNTRIES = set(DEBT_SOURCE_SWITCHES.keys())
# Countries with delayed sample entry in Table C1.
LATE_ENTRY_COUNTRIES = {c for c, y in SAMPLE_START_YEAR.items() if int(y) > 2003}
def _debt_source(country_iso3: str, year: int) -> str:
    sw = DEBT_SOURCE_SWITCHES.get(country_iso3)
    if sw is not None:
        return sw["after"] if int(year) >= int(sw["from_year"]) else sw["before"]
    return DEBT_SOURCE_BASE[country_iso3]
# --- Step A totals (residency) ---
# Performance mode: only compute TARGET_YEAR (final table year).
base_rows = []
for country_iso3 in COUNTRY_ORDER:
    year = int(TARGET_YEAR)
    debt_src = _debt_source(country_iso3, year)
    eq_src = EQUITY_SOURCE[country_iso3]
    if debt_src == "OECD":
        st_mn, lt_mn = _oecd_debt_st_lt_usd_million(country_iso3, year)
    else:
        st_mn, lt_mn = _bis_debt_st_lt_usd_million(country_iso3, year)
    if eq_src == "OECD":
        eq_mn = _oecd_equity_usd_million(country_iso3, year)
        if eq_mn is None:
            eq_mn = _msci_equity_spliced_usd_million(
                country_iso3,
                year,
                "OECD",
                lambda yy: _oecd_equity_usd_million(country_iso3, int(yy)),
            )
    else:
        eq_mn = _wb_equity_usd_million(country_iso3, year)
        if eq_mn is None:
            eq_mn = _msci_equity_spliced_usd_million(
                country_iso3,
                year,
                "WB",
                lambda yy: _wb_equity_usd_million(country_iso3, int(yy)),
            )
    base_rows.append(
        {
            "country": country_iso3,
            "year": int(year),
            "debt_source": debt_src,
            "equity_source": eq_src,
            "sample_start_year": int(SAMPLE_START_YEAR[country_iso3]),
            "st_mn_residency": st_mn,
            "lt_mn_residency": lt_mn,
            "eq_mn_residency": eq_mn,
        }
    )
base_panel = pd.DataFrame(base_rows)
# Appendix C: restate totals from issuer residency to nationality.
totals_long = base_panel.melt(
    id_vars=["country", "year", "debt_source", "equity_source", "sample_start_year"],
    value_vars=["st_mn_residency", "lt_mn_residency", "eq_mn_residency"],
    var_name="metric",
    value_name="value_mn",
)
metric_to_asset = {"st_mn_residency": "ST_DEBT", "lt_mn_residency": "LT_DEBT", "eq_mn_residency": "EQUITY"}
totals_long["asset_class"] = totals_long["metric"].map(metric_to_asset)
totals_long = totals_long.rename(columns={"country": "issuer", "year": "TIME_PERIOD"})
totals_long["value_usd"] = pd.to_numeric(totals_long["value_mn"], errors="coerce") * 1e6
restated_totals = _apply_restatement_by_issuer(totals_long[["issuer", "TIME_PERIOD", "asset_class", "value_usd"]])
restated_totals["value_mn"] = pd.to_numeric(restated_totals["value_usd"], errors="coerce") / 1e6
restated_totals_w = restated_totals.pivot_table(
    index=["issuer", "TIME_PERIOD"],
    columns="asset_class",
    values="value_mn",
    aggfunc="sum",
).reset_index()
restated_totals_w = restated_totals_w.rename(columns={"issuer": "country", "TIME_PERIOD": "year", "ST_DEBT": "st_mn", "LT_DEBT": "lt_mn", "EQUITY": "eq_mn"})
restated_totals_w["year"] = pd.to_numeric(restated_totals_w["year"], errors="coerce").astype("Int64")
panel = base_panel.merge(restated_totals_w[["country", "year", "st_mn", "lt_mn", "eq_mn"]], on=["country", "year"], how="left")
# Fill missing post-restatement with residency values when a country does not appear in matrix outputs.
panel["st_mn"] = panel["st_mn"].where(pd.notna(panel["st_mn"]), panel["st_mn_residency"])
panel["lt_mn"] = panel["lt_mn"].where(pd.notna(panel["lt_mn"]), panel["lt_mn_residency"])
panel["eq_mn"] = panel["eq_mn"].where(pd.notna(panel["eq_mn"]), panel["eq_mn_residency"])
# Local-currency debt diagnostic track (do not overwrite total debt outstanding).
st_local_vals = []
lt_local_vals = []
bis_missing_base_flags = []
oecd_negative_subtraction = []
for _, rr in panel.iterrows():
    iso2 = ISO3_TO_ISO2.get(str(rr["country"]))
    st_fc_mn = 0.0
    lt_fc_mn = 0.0
    if iso2 is not None and pd.notna(rr["year"]):
        st_fc_mn, lt_fc_mn = _bis_ids_foreign_st_lt_usd_million(iso2, int(rr["year"]))
    st_raw = pd.to_numeric(rr["st_mn"], errors="coerce")
    lt_raw = pd.to_numeric(rr["lt_mn"], errors="coerce")
    debt_src = str(rr.get("debt_source", ""))
    has_bis_base = pd.notna(rr.get("st_mn_residency")) or pd.notna(rr.get("lt_mn_residency"))
    missing_bis_base = debt_src == "BIS" and not has_bis_base
    bis_missing_base_flags.append(bool(missing_bis_base))
    # If BIS domestic base is missing, use instruction fallback:
    # domestic = total debt - international debt; split by IDS maturity shares.
    if missing_bis_base:
        total_raw = 0.0
        if pd.notna(st_raw):
            total_raw += float(st_raw)
        if pd.notna(lt_raw):
            total_raw += float(lt_raw)
        ids_st_mn, ids_lt_mn = _bis_ids_all_st_lt_usd_million(str(iso2), int(rr["year"])) if iso2 is not None else (0.0, 0.0)
        ids_tot = float(ids_st_mn) + float(ids_lt_mn)
        if iso2 is not None and pd.notna(rr["year"]):
            tot_st_mn, tot_lt_mn = _bis_total_st_lt_usd_million(str(iso2), int(rr["year"]))
            tot_all = float(tot_st_mn) + float(tot_lt_mn)
            dom_all = max(0.0, tot_all - ids_tot)
            if dom_all > 0.0 and ids_tot > 0.0:
                st_share_ids = float(ids_st_mn) / ids_tot
                dom_st = dom_all * st_share_ids
                dom_lt = dom_all * (1.0 - st_share_ids)
                st_local_vals.append(dom_st + float(ids_st_mn))
                lt_local_vals.append(dom_lt + float(ids_lt_mn))
                continue
            if tot_all > 0.0 and (tot_st_mn > 0.0 or tot_lt_mn > 0.0):
                st_local_vals.append(float(tot_st_mn))
                lt_local_vals.append(float(tot_lt_mn))
                continue
        if total_raw > 0.0 and ids_tot > 0.0:
            st_share_ids = float(ids_st_mn) / ids_tot
            st_local_vals.append(total_raw * st_share_ids)
            lt_local_vals.append(total_raw * (1.0 - st_share_ids))
        else:
            st_local_vals.append(None if pd.isna(st_raw) else float(st_raw))
            lt_local_vals.append(None if pd.isna(lt_raw) else float(lt_raw))
        continue
    st_local = None if pd.isna(st_raw) else float(st_raw)
    lt_local = None if pd.isna(lt_raw) else float(lt_raw)
    # Local-currency debt alignment: subtract BIS foreign-currency IDS on residency totals first,
    # then map to nationality using the same-year restatement ratio.
    if debt_src == "OECD":
        st_resid = pd.to_numeric(rr.get("st_mn_residency"), errors="coerce")
        lt_resid = pd.to_numeric(rr.get("lt_mn_residency"), errors="coerce")
        st_fc_adj = float(st_fc_mn)
        lt_fc_adj = float(lt_fc_mn)
        if pd.notna(st_resid) and float(st_fc_adj) > float(st_resid):
            oecd_negative_subtraction.append((str(rr.get("country", "")), "ST(strict-invalid)", float(st_resid), float(st_fc_adj)))
            st_fc_adj = float("nan")
        if pd.notna(lt_resid) and float(lt_fc_adj) > float(lt_resid):
            oecd_negative_subtraction.append((str(rr.get("country", "")), "LT(strict-invalid)", float(lt_resid), float(lt_fc_adj)))
            lt_fc_adj = float("nan")
        if pd.notna(st_resid):
            if pd.isna(st_fc_adj):
                st_local = float(st_raw) if pd.notna(st_raw) else float(st_resid)
            else:
                st_resid_adj = float(st_resid) - float(st_fc_adj)
                st_ratio = 1.0
                if pd.notna(st_raw) and float(st_resid) > 0.0:
                    st_ratio = float(st_raw) / float(st_resid)
                st_local = max(0.0, st_resid_adj) * st_ratio
        if pd.notna(lt_resid):
            if pd.isna(lt_fc_adj):
                lt_local = float(lt_raw) if pd.notna(lt_raw) else float(lt_resid)
            else:
                lt_resid_adj = float(lt_resid) - float(lt_fc_adj)
                lt_ratio = 1.0
                if pd.notna(lt_raw) and float(lt_resid) > 0.0:
                    lt_ratio = float(lt_raw) / float(lt_resid)
                lt_local = max(0.0, lt_resid_adj) * lt_ratio
    # Keep BIS totals on the same post-restatement chain; do not add IDS again here.
    st_local_vals.append(st_local)
    lt_local_vals.append(lt_local)
panel["st_local_mn"] = st_local_vals
panel["lt_local_mn"] = lt_local_vals
panel["bis_missing_base"] = bis_missing_base_flags
if oecd_negative_subtraction:
    print("WARNING: OECD-BIS subtraction negative on residency totals for:")
    for cc, mat, resid_v, bis_v in oecd_negative_subtraction:
        print(f"  {cc} {mat}: residency={resid_v:.3f} mn, bis_ids_foreign={bis_v:.3f} mn")
# Appendix C: foreign/reserve holdings are also restated from issuer residency to nationality.
def _attach_pip_sum(panel_df: pd.DataFrame, agg_df: pd.DataFrame, asset: str, out_col: str) -> pd.DataFrame:
    x = agg_df[agg_df["asset_class"].astype(str) == asset].copy()
    x["value_mn"] = pd.to_numeric(x["value_usd"], errors="coerce") / 1e6
    x = x.rename(columns={"issuer": "country", "TIME_PERIOD": "year"})[["country", "year", "value_mn"]]
    x["year"] = pd.to_numeric(x["year"], errors="coerce").astype("Int64")
    return panel_df.merge(x.rename(columns={"value_mn": out_col}), on=["country", "year"], how="left")
debt_foreign_agg = pip_foreign_local_agg if USE_LOCAL_FOREIGN_DEBT else pip_foreign_agg
panel = _attach_pip_sum(panel, debt_foreign_agg, "ST_DEBT", "foreign_st_mn")
panel = _attach_pip_sum(panel, debt_foreign_agg, "LT_DEBT", "foreign_lt_mn")
panel = _attach_pip_sum(panel, pip_foreign_agg, "EQUITY", "foreign_eq_mn")
panel = _attach_pip_sum(panel, pip_reserve_agg, "ST_DEBT", "reserve_st_mn")
panel = _attach_pip_sum(panel, pip_reserve_agg, "LT_DEBT", "reserve_lt_mn")
panel = _attach_pip_sum(panel, pip_reserve_agg, "EQUITY", "reserve_eq_mn")
if "bis_missing_base" in panel.columns:
    _m = panel["bis_missing_base"] == True
    panel.loc[_m, ["foreign_st_mn", "foreign_lt_mn", "reserve_st_mn", "reserve_lt_mn"]] = None
# Data-instruction alignment: SHC replacement is total-holdings based.
# Skip it when debt foreign inputs are local-currency allocated to avoid mixing definitions.
if not USE_LOCAL_FOREIGN_DEBT:
    usa_stlt = _usa_bilateral_st_lt_million()
    usa_lookup = {(str(r["issuer"]), str(r["asset_class"])): float(r["usa_mn"]) for _, r in usa_stlt.iterrows() if pd.notna(r["usa_mn"])}
    for asset, shc_path, out_col in [
        ("ST_DEBT", SHC_A7_PATH, "foreign_st_mn"),
        ("LT_DEBT", SHC_A8_PATH, "foreign_lt_mn"),
    ]:
        for i, row in panel.iterrows():
            iso3 = str(row["country"])
            shc_name = ISO3_TO_SHC_NAME.get(iso3, iso3)
            shc_val = _read_shc_country_total_million(shc_path, shc_name)
            if shc_val is None:
                continue
            foreign_before = row[out_col]
            if pd.isna(foreign_before):
                continue
            usa_before = usa_lookup.get((iso3, asset), 0.0)
            panel.at[i, out_col] = max(0.0, float(foreign_before) - float(usa_before) + float(shc_val))
panel["debt_billion_usd"] = (pd.to_numeric(panel["st_mn"], errors="coerce") + pd.to_numeric(panel["lt_mn"], errors="coerce")) / 1000.0
panel["st_debt_billion_usd"] = pd.to_numeric(panel["st_local_mn"], errors="coerce") / 1000.0
panel["lt_debt_billion_usd"] = pd.to_numeric(panel["lt_local_mn"], errors="coerce") / 1000.0
panel["equity_billion_usd"] = pd.to_numeric(panel["eq_mn"], errors="coerce") / 1000.0
# Keep missing foreign-local debt as missing; do not coerce to zero.
_st_foreign_for_share = pd.to_numeric(panel["foreign_st_mn"], errors="coerce")
_lt_foreign_for_share = pd.to_numeric(panel["foreign_lt_mn"], errors="coerce")
panel["domestic_share_st_debt"] = [
    _domestic_share(t, f) for t, f in zip(panel["st_local_mn"], _st_foreign_for_share)
]
panel["domestic_share_lt_debt"] = [
    _domestic_share(t, f) for t, f in zip(panel["lt_local_mn"], _lt_foreign_for_share)
]
panel["domestic_share_equity"] = [
    _domestic_share(t, f) for t, f in zip(panel["eq_mn"], panel["foreign_eq_mn"])
]
_reserve_foreign_st = pd.to_numeric(panel["foreign_st_total_mn"], errors="coerce") if (USE_LOCAL_FOREIGN_DEBT and "foreign_st_total_mn" in panel.columns) else pd.to_numeric(panel["foreign_st_mn"], errors="coerce")
_reserve_foreign_lt = pd.to_numeric(panel["foreign_lt_total_mn"], errors="coerce") if (USE_LOCAL_FOREIGN_DEBT and "foreign_lt_total_mn" in panel.columns) else pd.to_numeric(panel["foreign_lt_mn"], errors="coerce")
panel["share_in_reserves_st_debt"] = [
    _share_in_reserves(r, f) for r, f in zip(panel["reserve_st_mn"], _reserve_foreign_st)
]
panel["share_in_reserves_lt_debt"] = [
    _share_in_reserves(r, f) for r, f in zip(panel["reserve_lt_mn"], _reserve_foreign_lt)
]
panel["share_in_reserves_equity"] = [
    _share_in_reserves(r, f) for r, f in zip(panel["reserve_eq_mn"], panel["foreign_eq_mn"])
]
full_country_panel = panel.sort_values(["country", "year"]).reset_index(drop=True)
asset_rows = []
for _, r in full_country_panel.iterrows():
    asset_rows.extend(
        [
            {
                "country": r["country"],
                "year": int(r["year"]),
                "asset_class": "Short-term debt",
                "value_billion_usd": r["st_debt_billion_usd"],
                "domestic_share": r["domestic_share_st_debt"],
                "share_in_reserves": r["share_in_reserves_st_debt"],
            },
            {
                "country": r["country"],
                "year": int(r["year"]),
                "asset_class": "Long-term debt",
                "value_billion_usd": r["lt_debt_billion_usd"],
                "domestic_share": r["domestic_share_lt_debt"],
                "share_in_reserves": r["share_in_reserves_lt_debt"],
            },
            {
                "country": r["country"],
                "year": int(r["year"]),
                "asset_class": "Equity",
                "value_billion_usd": r["equity_billion_usd"],
                "domestic_share": r["domestic_share_equity"],
                "share_in_reserves": r["share_in_reserves_equity"],
            },
        ]
    )
full_country_panel_long = pd.DataFrame(asset_rows)
panel_2020 = full_country_panel[full_country_panel["year"] == TARGET_YEAR].copy()
panel_2020_long = full_country_panel_long[full_country_panel_long["year"] == TARGET_YEAR].copy()


In [141]:
# Mark current table cells by deviation level
 # >40% => !! ; >10% => !
marker_map = {}
for _, rr in cmp_df.iterrows():
    rel = rr["RelDiff"]
    if pd.isna(rel):
        continue
    key = (rr["Issuer"], rr["Metric"])
    if rel > 0.40:
        marker_map[key] = "!!"
    elif rel > 0.10:
        marker_map[key] = "!"
marked = country_rows[["Issuer"] + metric_cols].copy()
for col in metric_cols:
    marked[col] = marked.apply(
        lambda r: f"{r[col]} {marker_map[(r['Issuer'], col)]}" if (r["Issuer"], col) in marker_map else r[col],
        axis=1,
    )
print("已在当前表中标注：>40% 为 !!，>10% 为 !")
display(marked)
marked_over10_table = marked

已在当前表中标注：>40% 为 !!，>10% 为 !


,Issuer,ST_Billion_USD,ST_Domestic_share,ST_Share_in_reserves,LT_Billion_USD,LT_Domestic_share,LT_Share_in_reserves,EQ_Billion_USD,EQ_Domestic_share,EQ_Share_in_reserves
CAN,Canada,521.0,0.95,0.0 !!,2467.0,0.79,0.03 !!,6576.0,0.87,0.01 !!
USA,United States,5497.0,0.98,0.1 !!,38725.0,0.92,0.13 !!,55635.0,0.89,0.04 !!
AUT,Austria,17.0,0.78 !!,0.21 !!,560.0,0.56 !,0.07 !,286.0,0.78,0.0
BEL,Belgium,124.0,0.83 !,0.13 !,1150.0,0.71 !,0.06,1737.0,0.93,0.01 !!
FIN,Finland,65.0,0.87 !!,0.29 !!,398.0,0.61 !!,0.09,657.0,0.76,0.01 !!
FRA,France,628.0,0.92 !,0.04 !!,5084.0,0.74 !,0.08 !,10411.0,0.89,0.02 !!
DEU,Germany,310.0,0.83 !!,0.07 !!,4511.0,0.83 !!,0.07 !!,3696.0,0.73 !,0.02 !!
ITA,Italy,170.0,0.66,0.21 !!,3752.0,0.83,0.09 !!,2206.0,0.89,0.01 !!
NLD,Netherlands,72.0,0.54 !,0.05 !!,1235.0,0.24 !!,0.15 !!,6204.0,0.88,0.01 !!
PRT,Portugal,26.0,0.82 !,0.18 !!,343.0,0.74,0.14 !!,386.0,0.92,0.01 !!
